<center>
<img src="../images/logo-poster-transparent.png" width="400px" style="opacity:0.7">
</center>

In [23]:
%run supportvectors-common.ipynb


<div style="color:#aaa;font-size:8pt">
<hr/>
&copy; SupportVectors AI Lab. <blockquote>This notebook is part of the AI Lab Book companion repository. 
Code is released under the MIT License and accompanying text under CC BY 4.0 - see the LICENSE files at the repository root.

 </blockquote>
 <hr/>
</div>



## 1. Environment Setup

**Install required libraries** (run this once in your environment).

This installs:
- `deepagents` - The DeepAgents framework
- `langchain-openai` - For OpenAI models (GPT-5, GPT-5-mini)
- `tavily-python` - For web search functionality
- `langgraph` - The underlying graph framework
- `python-dotenv` - For loading environment variables from `.env`

## 2. DeepAgents Overview & Basic Harness

**Framework vs harness (one mental model):**

| Layer | Role |
|---|---|
| **LangChain** | Building blocks — models, tools, messages |
| **LangGraph** | Runtime — durable state, streaming, interrupts |
| **Deep Agents** | Harness — opinionated stack on top of both |

A plain agent loop calls tools until it stops. A *harness* also gives that loop planning, a virtual filesystem, subagent delegation, summarization, and human approval hooks — so long-running work is more reliable without you wiring each piece by hand.

DeepAgents support:
- Multi-step planning
- Tool use
- Subagents (delegation)
- Filesystem integration
- Durable files (via backends) and always-on project memory (via `memory=` — notebook 02)
- Human-in-the-loop via interrupts

What you get without configuring: A bare create_deep_agent(model=...) already includes filesystem tools (ls, read_file, write_file, …), conversation summarization / context offloading, and tool-call repair. Pass subagents=[...] and the harness also adds the task tool so the main agent can delegate to those subagents. Skills, memory= / AGENTS.md, and permissions= are optional — see notebook 02.

We will start with a minimal research agent using the harness.

### 2.0 Model Information

**Current Model:** gpt-5-mini
- **Cost:** check in openai api pricing for latest prices
- **Best for:** Most tasks, good balance of speed and quality
- **Monitor usage:** https://platform.openai.com/usage


To change models, modify the `model` parameter in `ChatOpenAI`:
```python
model = ChatOpenAI(model="gpt-5-mini", timeout=300.0)
```


In [24]:
import os
from typing import Literal

from deepagents import create_deep_agent
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from tavily import TavilyClient
from langchain_openrouter import ChatOpenRouter


# Load environment variables from .env file
load_dotenv()
print("✅ Loaded environment variables")
print(f"   OPENAI_API_KEY: {'Set' if os.environ.get('OPENAI_API_KEY') else 'Not set'}")
print(f"   TAVILY_API_KEY: {'Set' if os.environ.get('TAVILY_API_KEY') else 'Not set'}")

# Use this if you want to use Openai Model
model = ChatOpenAI(model="gpt-5-mini", timeout=300.0,)

# Use this if you want to use Models hosted in SV cluster
# model =ChatOpenAI(model="openai/gpt-oss-20b", base_url="http://10.0.10.70:8000/v1", api_key="dummy")

# Use this if you want to use Models hosted in OpenRouter
# model = ChatOpenRouter(model="deepseek/deepseek-v4.1-flash-1017")


# Configure API keys via environment or here
# os.environ["TAVILY_API_KEY"] = "YOUR_TAVILY_KEY"


# Simple web search tool
_tavily = TavilyClient(api_key=os.environ.get("TAVILY_API_KEY", ""))


def internet_search(query: str,
                     max_results: int = 5,
                     topic: Literal["general", "news", "finance"] = "general",
                     include_raw_content: bool = False):
    """Run a web search with Tavily.

    Returns search results or an error message if API key is not set.
    """
    if not _tavily.api_key:
        return "TAVILY_API_KEY not set. Skipping web search."
    return _tavily.search(
        query,
        max_results=max_results,
        include_raw_content=include_raw_content,
        topic=topic,
    )


✅ Loaded environment variables
   OPENAI_API_KEY: Set
   TAVILY_API_KEY: Set


### 2.0.1 Basic Agent Parameters

The `create_deep_agent` function accepts several parameters to configure your agent. Here's what each parameter does:

| Parameter | Type | Description |
|-----------|------|-------------|
| `model` | `BaseChatModel` | **Required.** The LLM that powers the agent (e.g., `ChatOpenAI`, `ChatAnthropic`) |
| `tools` | `list` | Functions/tools the agent can call to perform actions |
| `system_prompt` | `str` | Instructions that define the agent's behavior, personality, and capabilities |
| `subagents` | `list[dict]` | Subagent configurations for delegating specialized tasks |
| `backend` | `BackendProtocol` | Filesystem backend instance (`StateBackend()`, `FilesystemBackend(...)`, etc.) |
| `store` | `BaseStore` | LangGraph store — needed with `StoreBackend`; this notebook uses `FilesystemBackend` for `/memories/` instead |
| `checkpointer` | `Checkpointer` | Enables conversation persistence and human-in-the-loop workflows |
| `middleware` | `list` | Extra hooks around the agent loop (retries, planning tools, guardrails, …) — see §2.0.2 |
| `interrupt_on` | `dict` | Specifies which tools require human approval before execution |

**Minimal Agent (just model):**
```python
agent = create_deep_agent(model=model)
```

**Common Configuration (model + tools + prompt):**
```python
agent = create_deep_agent(
    model=model,                    # The LLM to use
    tools=[internet_search],        # Tools the agent can call
    system_prompt="You are a..."    # Agent instructions
)
```

> 💡 **Tip:** Start simple with just `model`, `tools`, and `system_prompt`. Add other parameters as your use case requires more advanced features.


### 2.0.2 Middleware (one mental model)

[Middleware](https://docs.langchain.com/oss/python/langchain/middleware/overview) is how you control what happens *inside* the agent loop — hooks that run before/after the model call and before/after tools. Use it for retries, guardrails, human approval, planning tools, logging, and similar cross-cutting behavior.

You add middleware by passing a list to `create_deep_agent` (same pattern as LangChain's `create_agent`):

```python
from langchain.agents.middleware import TodoListMiddleware

agent = create_deep_agent(
    model=model,
    tools=[internet_search],
    system_prompt="You are a helpful assistant.",
    middleware=[TodoListMiddleware()],  # opt-in: write_todos / read_todos
)
```

Deep Agents already ships a default middleware stack (filesystem tools, summarization, tool-call repair, and more). Your `middleware=[...]` list **adds** pieces — or, if an instance's `.name` matches a built-in, **replaces** that piece.

In this notebook you will meet:
- `FilesystemMiddleware` in #5 (trim which file tools the model sees for the HITL demo)
- `TodoListMiddleware` in #6 (opt-in planning tools)

Notebook 02 adds `ToolRetryMiddleware` for flaky tools.

### 2.1 Try the Basic Agent

Ask a simple research question to see the harness in action.

In [25]:
# System prompt to steer the agent to be an expert researcher
research_instructions = """You are an expert researcher. Your job is to conduct thorough research and then write a polished report.

You have access to an internet search tool as your primary means of gathering information.

## `internet_search`

Use this to run an internet search for a given query. You can specify the max number of results to return, the topic, and whether raw content should be included.
"""

agent = create_deep_agent(
    model=model,
    tools=[internet_search],
    system_prompt=research_instructions
)

In [26]:
result = agent.invoke({"messages": [{"role": "user", "content": "What is LangGraph?"}]})

# What did the harness actually call? (often internet_search)
tools_used = []
for msg in result["messages"]:
    for tc in getattr(msg, "tool_calls", None) or []:
        name = tc.get("name") if isinstance(tc, dict) else getattr(tc, "name", "?")
        tools_used.append(name)
print("Tools used:", tools_used or "(none — answered from model knowledge)")
print()
print(result["messages"][-1].content)


Tools used: ['internet_search']

Short answer
LangGraph is an open-source, low-level orchestration framework and runtime for building, running, and deploying long-running, stateful AI agents and workflows. It models agent logic as directed graphs (nodes = steps/actions, edges = control flow) and provides durable execution, shared state, streaming, human-in-the-loop, observability, and production deployment tooling. LangGraph is built by LangChain Inc., is MIT-licensed, and is designed to complement (not replace) higher-level tools like LangChain.

What it is (expanded)
- Graph-based orchestration: Workflows are expressed as directed graphs. Each node performs a unit of work (a deterministic function, a tool call, or an LLM-driven step). Edges encode control flow (including loops, branching, and parallel paths).
- Shared, durable state: Nodes read and update a shared state object that represents the workflow’s memory. State can be persisted so agents can be long-running and resume after

## 3. Filesystem Backends

DeepAgents expose filesystem tools (`ls`, `read_file`, `write_file`, `edit_file`, `glob`, `grep`)
backed by pluggable backends. Pick the backend by how long the file should live:

| Backend | Lives where | Survives |
|---|---|---|
| `StateBackend` | agent graph state | this thread only |
| `FilesystemBackend` | real folder under `root_dir` | process / kernel restart |
| `StoreBackend` | LangGraph store | across threads (for the lifetime of that store) |
| `CompositeBackend` | router | routes path prefixes to different backends |

We use a `CompositeBackend`: scratch paths stay on `StateBackend`; `/memories/` goes to `FilesystemBackend` so files land on disk under the project's `memories/` folder.

**`thread_id` vs `/memories/`:** `thread_id` scopes *conversation history* (chat turns in this run). Files under `/memories/` are shared across threads because of the backend — that is why `thread-2` can still read what `thread-1` wrote.

**Two different “memory” ideas (do not mix them up):**

1. **Durable files** under `/memories/` (this section) — the agent *writes* artifacts with `write_file`; they persist according to the backend.
2. **Always-loaded context** via `memory=[".../AGENTS.md"]` — standing instructions injected into the system prompt at startup (notebook 02).

**Note**: With `FilesystemBackend`, stale data is a real file on disk (e.g. `memories/favorite_color.txt`). Delete that file, or change the path, if you want a clean run. Restarting the kernel alone will not clear disk-backed memories.


In [27]:
from pathlib import Path

from deepagents import create_deep_agent
from deepagents.backends import CompositeBackend, StateBackend, FilesystemBackend

# Prefer BOOTCAMP_ROOT_DIR; otherwise walk up from cwd looking for pyproject.toml
# so memories/ lands at the project root even if the notebook cwd is docs/notebooks/.
if os.environ.get("BOOTCAMP_ROOT_DIR"):
    project_root = Path(os.environ["BOOTCAMP_ROOT_DIR"]).resolve()
else:
    project_root = Path.cwd().resolve()
    for candidate in [project_root, *project_root.parents]:
        if (candidate / "pyproject.toml").exists():
            project_root = candidate
            break

memories_dir = project_root / "memories"
memories_dir.mkdir(parents=True, exist_ok=True)
print(f"memories_dir → {memories_dir}")

# Scratch paths stay on StateBackend (ephemeral).
# /memories/ is routed to real disk under memories/ so it survives kernel restarts.
composite_backend = CompositeBackend(
    default=StateBackend(),
    routes={
        "/memories/": FilesystemBackend(root_dir=memories_dir, virtual_mode=True),
    },
)

instructions = """
When the user says: "My favorite color is X":
- Extract just the color X.
- Use write_file to save it to `/memories/favorite_color.txt`.

When the user asks: "What is my favorite color?":
- Try to read `/memories/favorite_color.txt` using read_file.
- If found, reply: "Your favorite color is <COLOR>."
- If not found, reply: "I don't know your favorite color yet."

Use the filesystem tools for reading, writing, and listing files when relevant.
"""

agent = create_deep_agent(
    model=model,
    backend=composite_backend,
    system_prompt=instructions,
)


memories_dir → /Users/ashwathbhat/Documents/GitHub/ai_agents_bootcamp/deepagents_prda/memories


In [28]:
from langchain_core.runnables import RunnableConfig

config_thread_1: RunnableConfig = {"configurable": {"thread_id": "thread-1"}}

print("=== THREAD 1: Saving favorite color ===")
result_1 = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "My favorite color is Red."}
        ]
    },
    config=config_thread_1,
)

# Show that file really exists in the agent's virtual FS
result_1_ls = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "List /memories."}
        ]
    },
    config=config_thread_1,
)
print(result_1_ls["messages"][-1].content)

# Proof outside the agent: same bytes on disk under the project
on_disk = memories_dir / "favorite_color.txt"
print()
print(f"On disk: {on_disk}")
print(f"Exists:  {on_disk.exists()}")
if on_disk.exists():
    print(f"Contents: {on_disk.read_text(encoding='utf-8')!r}")


=== THREAD 1: Saving favorite color ===
I see one file in /memories:
- /memories/favorite_color.txt

Would you like me to read it?

On disk: /Users/ashwathbhat/Documents/GitHub/ai_agents_bootcamp/deepagents_prda/memories/favorite_color.txt
Exists:  True
Contents: 'Red'


In [29]:
from langchain_core.runnables import RunnableConfig

# Fresh thread_id → no chat history from thread-1, but /memories/ file is still there
config_thread_2: RunnableConfig = {"configurable": {"thread_id": "thread-2"}}

print("\n=== THREAD 2: Reading favorite color (no prior chat in this thread) ===")
result_2 = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "What is my favorite color?"}
        ]
    },
    config=config_thread_2,
)

print(result_2["messages"][-1].content)



=== THREAD 2: Reading favorite color (no prior chat in this thread) ===
Your favorite color is Red.


## 4. Subagents

Subagents let the main agent delegate work to helper agents with:
- Their own system prompts
- Their own tools
- Their own (clean) context

Use them for context quarantine and specialization.

In [30]:
import os
from typing import Literal

from deepagents import SubAgent, create_deep_agent
from tavily import TavilyClient

# -----------------------------
# Tool: Internet search (Tavily)
# -----------------------------
tavily_client = TavilyClient(api_key=os.environ.get("TAVILY_API_KEY", ""))

def internet_search(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news", "finance"] = "general",
    include_raw_content: bool = False,
):
    """Run a web search using Tavily (mock result if no key is set)."""
    if not tavily_client.api_key:
        return f"[MOCK SEARCH RESULT] {query}: example findings A, B, C."
    return tavily_client.search(
        query=query,
        max_results=max_results,
        include_raw_content=include_raw_content,
        topic=topic,
    )

# -----------------------------
# Subagent: Research agent
# -----------------------------
research_subagent: SubAgent = {
    "name": "research-agent",
    "description": "Performs in-depth research on a topic using web search.",
    "system_prompt": """
You are a focused research agent.

- Your job is to perform in-depth research when requested.
- Use the `internet_search` tool to gather up-to-date information.
- Summarize the findings clearly and concisely for the main agent.
- Prefer bullet points and short paragraphs over long essays.
""",
    "tools": [internet_search],
    # Optional per-subagent model override; omit to inherit the main agent model
    "model": "openai:gpt-5-mini",
}

subagents = [research_subagent]

# -----------------------------
# Main Orchestrator Agent
# -----------------------------
main_instructions = """
You are an orchestration agent.

Behavior:
- For simple, timeless questions (definitions, basic explanations, overviews),
  answer directly without delegating.
- For questions that mention things like:
  - "latest", "recent", "current"
  - "research", "trends", "in-depth analysis"
  - or explicitly ask to "use web" / "search online"
  you SHOULD delegate to the `research-agent` subagent.

When you delegate:
- Pass the user question and relevant context to the research-agent.
- Let the research-agent call tools (like `internet_search`) as needed.
- When the research-agent returns, synthesize and present a clean final answer
  to the user in your own words.

Always be explicit, structured, and easy to read.
"""

agent = create_deep_agent(
    model=model,
    system_prompt=main_instructions,
    subagents=subagents,
)


### 4.1 Try Delegation (Conceptual)

In practice, the main agent decides when to use the `task` tool to call subagents.

We run two prompts: a timeless definition (answer directly) and a "recent research" question (should delegate). Watch which run lists `task` in **Tools used**.


In [31]:
from langchain_core.runnables import RunnableConfig


def tools_used(result) -> list[str]:
    names: list[str] = []
    for msg in result["messages"]:
        for tc in getattr(msg, "tool_calls", None) or []:
            name = tc.get("name") if isinstance(tc, dict) else getattr(tc, "name", "?")
            names.append(name)
    return names


config: RunnableConfig = {"configurable": {"thread_id": "poc-subagent-demo"}}

print("=== RUN A: timeless question (should NOT use research-agent) ===")
result_direct = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is a qubit? Answer in two sentences.",
            }
        ]
    },
    config=config,
)
print("Tools used:", tools_used(result_direct))
print(result_direct["messages"][-1].content)

print("\n=== RUN B: research question (should use research-agent via task) ===")
result_research = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "What are some recent research trends in quantum computing? "
                    "Give me 5 key areas with 1–2 lines each, based on up-to-date web research."
                ),
            }
        ]
    },
    config=config,
)
print("Tools used:", tools_used(result_research))
print(result_research["messages"][-1].content)


=== RUN A: timeless question (should NOT use research-agent) ===
Tools used: []
A qubit is the basic unit of quantum information: a two-level quantum system (for example, an electron spin or photon polarization) that can be in a superposition of the |0> and |1> states. Measuring a qubit yields a definite 0 or 1 and typically collapses its superposition, and qubits can become entangled with one another to produce the nonclassical correlations that power quantum computing.

=== RUN B: research question (should use research-agent via task) ===
Tools used: ['task']
I searched recent literature and major lab posts and distilled five prominent, up-to-date research trends in quantum computing. Each item has a 1–2 line summary and 1–2 supporting citations.

1) Error correction / logical qubits  
   - Summary: Demonstrations pushing quantum error-correction toward and below fault-tolerance thresholds (surface-code progress and longer-lived logical qubits) are central to making large-scale, reli

### 4.2 Streaming progress (main agent + subagents)

`invoke` waits for the full answer. For long runs you want a live log of what the agent is doing — planning, tool calls, and work *inside* a subagent.

LangGraph streaming gives that. Pass `stream_mode="updates"` and `subgraphs=True` so events from delegated subagents surface too (without `subgraphs=True` you only see the orchestrator). This is the same pattern the PRDA Streamlit app uses for its status log.

Run the cell below after the subagent agent from §4 is still in memory. Watch for lines tagged `orchestrator` vs `subagent`.

In [32]:
from uuid import uuid4
from langchain_core.runnables import RunnableConfig

stream_config: RunnableConfig = {"configurable": {"thread_id": str(uuid4())}}

print("=== STREAMING: research via subagent (subgraphs=True) ===\n")

for item in agent.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "In 3 short bullets, what are current research trends "
                    "in quantum computing? Use the research-agent."
                ),
            }
        ]
    },
    config=stream_config,
    stream_mode="updates",
    subgraphs=True,  # surface events from inside the subagent
):
    namespace, update = item if isinstance(item, tuple) else ((), item)
    where = "orchestrator" if not namespace else "subagent"

    if not isinstance(update, dict):
        continue
    for node_update in update.values():
        if not isinstance(node_update, dict):
            continue
        for msg in node_update.get("messages", []) or []:
            name = getattr(msg, "name", None) or type(msg).__name__
            tool_calls = getattr(msg, "tool_calls", None) or []
            if tool_calls:
                for tc in tool_calls:
                    tc_name = tc.get("name") if isinstance(tc, dict) else getattr(tc, "name", "?")
                    print(f"[{where}] tool → {tc_name}")
            elif getattr(msg, "content", None):
                preview = str(msg.content).replace("\n", " ")[:100]
                if preview.strip():
                    print(f"[{where}] {name}: {preview}")

print("\n(Done — same agent as invoke, but you saw the path it took.)")

=== STREAMING: research via subagent (subgraphs=True) ===

[orchestrator] tool → task
[subagent] tool → internet_search
[subagent] internet_search: {"query": "quantum error correction logical qubit 2026 arXiv IBM Quantinuum 2025 2026", "follow_up_q
[subagent] tool → internet_search
[subagent] internet_search: {"query": "quantum internet 2026 entanglement satellite quantum network 2025 2026 progress arXiv pre
[subagent] tool → internet_search
[subagent] internet_search: {"query": "quantum error mitigation 2026 review arXiv 2025 2026 variational algorithms noise mitigat
[subagent] research-agent: [{'id': 'rs_076d58c8eb54e25a006ad775ee254087d1bba45b2fd35b2b61', 'summary': [], 'type': 'reasoning',
[orchestrator] task: {   "bullets": [     {       "summary": "Error correction & logical qubits: groups are successfully 
[orchestrator] AIMessage: - Error correction and logical qubits: researchers are demonstrating encoded, protected “logical” qu

(Done — same agent as invoke, but you saw the p

## 5. Human-in-the-loop (HITL)

For sensitive tools (delete_file, send_email, run_sql, etc.),
we can require human approval using `interrupt_on`.

This uses LangGraph interrupts and requires a checkpointer.

In [33]:
from deepagents import FilesystemMiddleware
from langchain.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command


@tool
def delete_file(path: str) -> str:
    """Mock delete. Call this immediately when asked to delete a file. Do not search for the file first."""
    return f'Deleted {path}'


@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Mock email. Call this immediately when asked to send an email."""
    return f'Sent email to {to}'


hitl_checkpointer = MemorySaver()

# The built-in filesystem tools include a real `delete`, plus ls/glob/execute.
# Those handle "Delete temp.txt" before the mock is ever called, so interrupt_on
# never fires. Keep only the required read_file; the prompt forces the mock.
hitl_agent = create_deep_agent(
    model=model,
    tools=[delete_file, send_email],
    system_prompt=(
        "When the user asks to delete a file, call delete_file immediately "
        "with the path they gave. When they ask to send an email, call send_email "
        "immediately. Do not read or search the workspace, delegate, plan, or reply "
        "with shell commands. These tools are mocks and succeed even if the file is not on disk."
    ),
    middleware=[FilesystemMiddleware(tools=["read_file"])],
    interrupt_on={
        'delete_file': True,
        'send_email': {'allowed_decisions': ['approve', 'edit', 'reject']},
    },
    checkpointer=hitl_checkpointer,
)

print('HITL-enabled agent created.')

HITL-enabled agent created.


### 5.1 HITL Flow Skeleton

1. Call `invoke`
2. If `__interrupt__` is present, read pending actions
3. Collect human decisions
4. Resume with `Command(resume={...})`

Below is a minimal skeleton (replace manual decisions with UI logic in real apps).

In [34]:
from uuid import uuid4

from langchain_core.runnables import RunnableConfig

print("=" * 70)
print("HUMAN-IN-THE-LOOP (HITL) DEMONSTRATION")
print("=" * 70)
print("\n📝 Scenario: Agent wants to delete a file")
print("⚠️  Sensitive operation requires human approval\n")

cfg: RunnableConfig = {"configurable": {"thread_id": str(uuid4())}}
result = hitl_agent.invoke({
    'messages': [{'role': 'user', 'content': 'Delete temp.txt'}]
}, config=cfg)

if result.get('__interrupt__'):
    interrupts = result['__interrupt__'][0].value
    action_requests = interrupts['action_requests']
    review_configs = interrupts['review_configs']

    print("🛑 AGENT PAUSED - APPROVAL REQUIRED")
    print("=" * 70)
    print('\n📋 Pending actions:')
    for action in action_requests:
        print(f'   • Tool: {action["name"]}')
        print(f'     Args: {action["args"]}')

    print("\n💭 In a real application, you would show this to a human for approval.")
    print("   For this demo, we'll auto-approve.\n")

    # Demo: auto-approve (in reality, ask a human)
    decisions = [{'type': 'approve'} for _ in action_requests]

    print("✅ Approval granted - resuming agent...\n")

    result = hitl_agent.invoke(
        Command(resume={'decisions': decisions}),
        config=cfg,
    )

print("=" * 70)
print("FINAL RESULT")
print("=" * 70)
print()
print('Agent response:', result['messages'][-1].content)
print()
print("=" * 70)
print("\n💡 Key Learning: Sensitive operations can require human approval!")


HUMAN-IN-THE-LOOP (HITL) DEMONSTRATION

📝 Scenario: Agent wants to delete a file
⚠️  Sensitive operation requires human approval

🛑 AGENT PAUSED - APPROVAL REQUIRED

📋 Pending actions:
   • Tool: delete_file
     Args: {'path': 'temp.txt'}

💭 In a real application, you would show this to a human for approval.
   For this demo, we'll auto-approve.

✅ Approval granted - resuming agent...

FINAL RESULT

Agent response: Done — temp.txt was deleted.


💡 Key Learning: Sensitive operations can require human approval!


### 5.2 Decision payloads *(reference — optional in live session)*

Live teaching can stop after approve/resume in §5.1. This cell documents the other decision types.

The resume call is the same for every decision. Put one decision per pending action, in the same order as `action_requests`:

```python
hitl_agent.invoke(
    Command(resume={"decisions": decisions}),
    config=cfg,
)
```

`delete_file` is configured with `True`, so all four decisions are allowed. `send_email` allows only `approve`, `edit`, and `reject`.

**Approve** runs the tool call unchanged.

```python
decisions = [{"type": "approve"}]
```

**Edit** replaces the tool name and arguments, keeps the original tool-call id, then runs the tool.

```python
decisions = [{
    "type": "edit",
    "edited_action": {
        "name": "delete_file",
        "args": {"path": "other.txt"},
    },
}]
```

For `send_email`, `edited_action["args"]` is `to`, `subject`, and `body`.

**Reject** does not run the tool. The model receives an error tool message. `message` is optional; when omitted, the model is told not to retry the call unless the user asks.

```python
decisions = [{"type": "reject", "message": "Do not delete temp.txt."}]
```

**Respond** does not run the tool. `message` is required and is returned as a successful tool result.

```python
decisions = [{
    "type": "respond",
    "message": "temp.txt was already removed yesterday.",
}]
```


## 6. Planning & TODO Tools (Intuition)

**Key Concept:** DeepAgents can plan complex work with TODO lists — but in this build you opt in by passing `TodoListMiddleware()`.

### Why Planning Matters:

**Without Planning:**
- Agent tries to do everything at once
- Easy to miss steps
- Hard to track progress
- Difficult to resume if interrupted

**With Planning:**
- Breaks down complex tasks systematically
- Tracks progress through each step
- Ensures all requirements are met
- Makes agent reasoning transparent

### How It Works:

With `middleware=[TodoListMiddleware()]`, the agent gets:
- `write_todos` — create and update todo lists (merge=true to update, merge=false to replace)
- `read_todos` — read the current todo list

The agent decides when to use these based on task complexity. Simple questions skip planning; multi-step research usually does not.


### 6.1 Simple Task - No Planning Needed

Let's start with a simple task that doesn't require planning:


In [35]:
from deepagents import create_deep_agent
from langchain.agents.middleware import TodoListMiddleware

planning_instructions = """You are a helpful assistant.

    For simple tasks, answer directly.
    For complex multi-step tasks, create a plan first using TODO tools,
    then work through each step systematically."""
# This deepagents build does not install write_todos unless asked.
planning_agent = create_deep_agent(
    model=model,
    tools=[internet_search],
    system_prompt=planning_instructions,
    middleware=[TodoListMiddleware()],
    subagents=[],  # No subagents - focus on planning only
)

# Simple task - agent will answer directly without planning
print("=== SIMPLE TASK (No Planning Needed) ===\n")
result = planning_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "What is 2 + 2?"
    }]
})

print(result["messages"][-1].content)
print("\n💡 For simple tasks, the agent answers directly without creating a plan.")


=== SIMPLE TASK (No Planning Needed) ===

2 + 2 = 4.

💡 For simple tasks, the agent answers directly without creating a plan.


### 6.2 Visualizing the TODO List in Action

**Key learning:** see the agent's internal plan evolve — create, update status, complete.

The next cell builds a planner with `TodoListMiddleware()`, runs a multi-step research task, then extracts every `write_todos` call so you can watch the list change.


In [36]:
import json
from typing import Any

print("=" * 70)
print("📋 VISUALIZING TODO LIST - LIVE DEMO")
print("=" * 70)

# Create an agent that will use planning
visual_planner = create_deep_agent(
    model=model,
    tools=[internet_search],
    system_prompt="""You are a systematic researcher.

    For complex tasks:
    1. Create a TODO list using write_todos
    2. Work through each item step by step
    3. Mark items complete as you finish them
    4. Provide a final summary

    Be methodical and organized.""",
    middleware=[TodoListMiddleware()],
    subagents=[],  # No subagents - we're demonstrating planning tools only
)

# Give it a multi-step task
print("\n🎯 Task: Research 3 key features of Rust programming language\n")
print("⏳ Running agent...\n")

result = visual_planner.invoke({
    "messages": [{
        "role": "user",
        "content": "Research the top 3 key features that make Rust popular. For each feature, provide a 2-sentence explanation."
    }]
})

print("=" * 70)
print("🔍 EXTRACTING TODO LIST CHANGES")
print("=" * 70)

# Track TODO list changes
todo_history: list[dict[str, Any]] = []

for i, msg in enumerate(result["messages"]):
    # Look for write_todos tool calls
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tool_call in msg.tool_calls:
            if tool_call.get("name") == "write_todos":
                todos_data = tool_call.get("args", {}).get("todos", [])
                merge = tool_call.get("args", {}).get("merge", False)

                todo_history.append({
                    "step": len(todo_history) + 1,
                    "action": "UPDATE" if merge else "CREATE",
                    "todos": todos_data
                })

# Display TODO list evolution
if todo_history:
    print("\n📊 TODO LIST EVOLUTION:\n")

    for entry in todo_history:
        print(f"{'='*70}")
        print(f"STEP {entry['step']}: {entry['action']} TODO LIST")
        print(f"{'='*70}\n")

        for todo in entry['todos']:
            status = todo.get('status', 'unknown')
            content = todo.get('content', 'No content')

            # Visual status indicator
            if status == 'completed':
                icon = '✅'
            elif status == 'in_progress':
                icon = '🔄'
            elif status == 'pending':
                icon = '⏳'
            else:
                icon = '❓'

            print(f"{icon} [{status.upper()}] {content}")

        print()

    # Show final summary
    print(f"{'='*70}")
    print("📈 FINAL TODO STATUS")
    print(f"{'='*70}\n")

    final_todos = todo_history[-1]['todos']
    completed = sum(1 for t in final_todos if t.get('status') == 'completed')
    total = len(final_todos)

    print(f"Total tasks: {total}")
    print(f"Completed: {completed}")
    print(f"Progress: {completed}/{total} ({100*completed//total if total > 0 else 0}%)")

else:
    print("\n⚠️  No TODO list found - task may have been too simple for planning")

print("\n" + "=" * 70)
print("💡 KEY LEARNING")
print("=" * 70)
print("✅ You can see EXACTLY how the agent breaks down tasks")
print("✅ TODO items progress from ⏳ pending → 🔄 in_progress → ✅ completed")
print("✅ This transparency helps you understand agent reasoning!")
print("✅ Great for debugging and building trust with users")


📋 VISUALIZING TODO LIST - LIVE DEMO

🎯 Task: Research 3 key features of Rust programming language

⏳ Running agent...

🔍 EXTRACTING TODO LIST CHANGES

📊 TODO LIST EVOLUTION:

STEP 1: CREATE TODO LIST

🔄 [IN_PROGRESS] Select top 3 Rust features to present
⏳ [PENDING] Write 2-sentence explanations for each feature
⏳ [PENDING] Review and finalize output

STEP 2: CREATE TODO LIST

✅ [COMPLETED] Select top 3 Rust features to present
🔄 [IN_PROGRESS] Write 2-sentence explanations for each feature
⏳ [PENDING] Review and finalize output

STEP 3: CREATE TODO LIST

✅ [COMPLETED] Select top 3 Rust features to present
✅ [COMPLETED] Write 2-sentence explanations for each feature
🔄 [IN_PROGRESS] Review and finalize output

📈 FINAL TODO STATUS

Total tasks: 3
Completed: 2
Progress: 2/3 (66%)

💡 KEY LEARNING
✅ You can see EXACTLY how the agent breaks down tasks
✅ TODO items progress from ⏳ pending → 🔄 in_progress → ✅ completed
✅ This transparency helps you understand agent reasoning!
✅ Great for debugg

**Planning is most useful for:** multi-step research, pipelines with dependencies, report-style workflows.

**Planning is usually not needed for:** simple lookups, single tool calls, direct Q&A.

---

## Wrap-up

In this notebook you used the Deep Agents basics:

1. A minimal harness agent (`model` + `tools` + `system_prompt`)
2. Pluggable filesystem backends and durable `/memories/` files
3. Subagent delegation, plus live streaming with `subgraphs=True`
4. Human-in-the-loop interrupts and resume decisions
5. Optional TODO planning via `TodoListMiddleware()`

**Next:** notebook `02_deepagents_harness.ipynb` adds the production knobs — `skills=`, always-loaded `memory=` / AGENTS.md, `permissions=`, and tool retries. The PRDA Streamlit app in `src/deepagents_basics/` puts the full stack together as a product-research pipeline.
